<div align="left">

# 章节介绍：认识 PyPTO 算子开发 Agent

前四章是从零走通了手工开发 PyPTO 算子的完整路径：从数学定义到 PyTorch golden，从 kernel 设计到调试，最终在真实 Ascend NPU 上完成精度验证。本章引入 PyPTO 算子开发 Agent（简称 PyPTO Agent）——它把整套开发流程交给一组分工明确的智能体自动完成，学习者的角色从「亲手实现算子」转变为「定义任务、审核结果」。

本节先建立一个整体认识：PyPTO Agent 是什么，为什么仅靠单条 Prompt 让大模型独立开发算子不可行，以及本章的学习路径。

</div>

<div align="left">

## 学习目标与前置要求

### 前置能力与环境

开始本章前应满足以下条件：

- 已完成本课程前四章，理解 shape、dtype、Tile 与 golden 的基本作用；
- 能阅读简单的 PyTorch 参考实现与 PyPTO 实现；
- 已进入课程环境，其中 OpenCode、PyPTO、Torch NPU 与 Ascend NPU 均已装好并可用；PyPTO Agent 工程（技能库和编排器）需要按 05.02 的部署步骤准备。

OpenCode 是本章与 Agent 交互的统一入口。OpenCode 和 PyPTO 等基础工具已由课程环境装好，本章不需要再安装它们；唯一的准备工作是 05.02 会讲到的「下载并部署 Agent 工程」，照着做即可。

### 学习目标与验收边界

完成本章后，读者应能够：

1. 说明 PyPTO Agent 的整体架构、各智能体职责，以及保障结果可信的核心机制；
2. 将算子的数学定义、接口、支持范围与精度要求整理为 Agent 可执行的任务；
3. 使用 OpenCode 运行 PyPTO Agent，并识别规格、golden、实现与测试等关键产物；
4. 独立重跑生成的测试，依据真实 Ascend NPU、逐输出 all-close、明确容差与退出码作出验收结论。

本章的必做内容仅限功能与精度验证：生成的实现必须在真实 Ascend NPU 上与独立 golden 按 `atol=1e-3`、`rtol=1e-3` 逐输出比较通过，且测试进程退出码为 0。性能优化是精度通过后的可选工作，不属于本章完成条件。

> **说明**：本章示例使用较小的固定 shape（如 `[8,128]`、`[8,8]`）是为了便于初学者理解和快速验证。实际生产环境中，PyPTO Agent 同样支持动态 shape（使用 `pypto.DYNAMIC`），任务定义方式相同，只需将固定值替换为 `DYNAMIC`。

> **术语速览**：本章会反复用到下面这些词，先做一个简单说明。
> - **shape / dtype**：张量的形状和数据类型。`[8,128]` 表示一个 8 行 128 列的二维张量；`FP32` 是 32 位浮点数，即带小数部分的数，精度较高。
> - **Tile（分块）**：把大张量切成一个个小块（Tile），让 NPU 逐块处理，避免一次性占用过多片上存储。
> - **golden（参考实现）**：用 PyTorch 写的「标准答案」实现，用来和 Agent 或手写的实现比对，判断结果是否正确。
> - **Prompt（提示词）**：发给大模型的一段文字指令，描述你想让它做什么。
> - **Kernel（核函数）**：真正在 NPU 上运行的计算代码。
> - **NPU**：昇腾神经网络处理器的统称，用于加速 AI 计算。
> - **容差（atol / rtol）**：允许结果与参考之间有多大的误差。`atol` 是绝对误差，`rtol` 是相对误差。`all_close=true` 表示两者在所有位置上误差都在容差内、算通过。
> - **退出码**：程序结束时的返回值，`0` 表示正常结束。测试脚本退出码为 0，通常意味着测试全部通过。

</div>

<div align="left">

## 为什么不能仅依靠单条 Prompt 开发算子

大模型能力越来越强，一个自然会冒出的问题是：直接让它「开发一个 PyPTO 算子」，结果靠得住吗？工程实践给出的答案是：靠不太住。算子开发是一整条流水线——需求理解、参考实现、方案设计、编码、调试、验证，任何一个环节出错都会影响最终结果。由一个模型独自做完这条流水线时，容易出现下面四类典型问题，而且这些问题大多很隐蔽，不容易发现：

1. **信息过载，关键约束易被遗漏**。单一智能体需要同时承载全部开发知识与流程，极易遗忘需求、跳过规范或忽略关键约束。一个真实案例：即便是简单的 ReLU 算子，能力较弱的模型也曾生成 `pypto.Tensor([], pypto.DT_FP32)` 这样的空 Shape 张量声明——不符合规范、无法按 Shape 编译，但粗看与正常代码差异不大。
2. **编码与验证不分，结论不可采信**。如果同一个模型既是实现的编写者，又是与 golden 比对的校验者，它可能将 golden 简单封装后充当实现，再宣布「验证通过」。编写者与判定者为同一主体时，验证结果无法完全采信。
3. **缺少强制约束，步骤被悄悄跳过**。如果没有独立的检查机制盯着每一步，流程就可能被简化、规范被绕过——比如在实现文件里偷偷混入 Torch 代码。这类问题通常不会主动报错，但任务还是会被标记为「已完成」。
4. **缺乏状态持久化，中断后难以恢复**。若没有结构化的进度记录，一次执行失败可能污染后续环节，任务中断后无法从断点继续，完整过程也难以审计。

这些问题的共同特征是「隐蔽」：不会直接报错，肉眼也不易识别。因此，仅依靠更强的模型并不能从根本上解决问题，还需要一整套围绕模型的工程机制来发现、拦截并修复。

</div>

<div align="left">

## PyPTO Agent 的解决思路

PyPTO Agent 的思路很直白：不指望模型永远不出错，而是搭一个**只有合规的代码才能顺利走完全程**的环境。模型还是会犯错，但错误会被发现、被拦下、被要求修复。它有五个关键设计，正好一一对应上一节提到的四类问题：

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">设计</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">解决的问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">作用</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">智能体分工</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信息过载、关键约束遗漏</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">将长流程拆解为多个各管一段的智能体，每个智能体只需关注自身任务，无需一次承载全部知识</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立验证</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编码与验证不分</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写实现与结果校验由不同智能体承担，从机制上避免「自己写、自己判」</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Lint Gate（自动门禁）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">步骤被静默跳过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">每次写入关键文件时自动执行规则检查，违规代码直接被拦截，智能体无法跳过</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态持久化</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">中断后难以恢复</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">将当前阶段、重试次数与决策依据持久化保存，失败可恢复、过程可追溯</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">真实 NPU 验收</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">结果不可靠</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">最终以真实 Ascend NPU 上与独立 golden 的比对结果为依据，用数值证据说话</td></tr></tbody></table>

这一整套围绕模型的工程约束，在工程上统称为 harness（约束体系）。它的作用一句话就能说清：即使模型偶尔犯错、代码写得不合规，机制也能及时发现问题并拦下，保证最终交付的质量过关。例如前文提到的空 Shape ReLU，在 PyPTO Agent 中会在代码写入时就被自动检查拦下，必须修复后才能继续。

简单理解：Agent 是干活的，harness 是定规矩的，两者配合，才能既干得快、又靠得住。

对于学习者而言，使用 Agent 时最关键的两件事是：**给出清晰明确的任务定义**，以及**只采信独立验证的结果证据**。

</div>

<div align="left">

## 本章学习路径

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">小节</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">主题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">学习结果</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">章节介绍（05.01_chapter_intro.ipynb）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">认识 PyPTO Agent</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">理解智能体分工与工程机制的必要性</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Agent 架构与使用方法（05.02_agent_architecture_and_usage.ipynb）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">架构与使用</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">掌握 Agent 整体架构、支撑体系，以及 OpenCode 操作与独立验收方法</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">章节实践（05.03_chapter_practice.ipynb）</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">动手实践</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立完成行 Softmax 的 Agent 开发与真实 NPU 验收</td></tr></tbody></table>

</div>

<div align="left">

## 课后练习

本节练习用于复盘单一 Prompt 开发算子的局限与 PyPTO Agent 的工程保障思路。单选题只有一个正确答案，多选题有两个或以上正确答案，填空题请填写正确的术语。

1. （单选题）随着大模型能力持续提升，为什么 PyPTO Agent 仍然需要智能体分工、Lint Gate 与独立验证这些工程设计？  
   A. 不需要，模型足够强之后这些设计都可以去掉  
   B. 模型仍可能犯错，且错误往往隐蔽、不报错；这些设计负责发现缺陷、拦截违规、留存证据，保证交付结果的底线  
   C. 这些设计是为了替代模型编写 Kernel 代码  
   D. 这些设计只是为了让代码生成得更快
2. （单选题）在 PyPTO Agent 的五项关键设计中，哪一项直接解决了「编码与验证不分」的问题？  
   A. 智能体分工  
   B. Lint Gate（自动门禁）  
   C. 独立验证  
   D. 状态持久化
3. （多选题）以下哪些是「仅依靠单条 Prompt 开发算子」容易遇到的隐蔽问题？（选择所有适用项）  
   A. 信息过载，关键约束易被遗漏  
   B. 编码与验证不分，结论不可采信  
   C. 缺少强制约束，步骤被静默跳过  
   D. 缺乏状态持久化，中断后难以恢复  
   E. 模型生成的代码运行速度太慢
4. （填空题）四类隐蔽问题有一个共同特征：它们大多不会主动报错、肉眼也不易发现，因此仅靠更强的模型并不能根治，还需要一套____________来发现缺陷、拦截违规并留存证据。
5. （填空题）PyPTO Agent 的五个关键设计中，在写入关键文件时自动拦截违规代码的是____________，让编写实现与结果判定由不同主体承担的是____________。
6. （单选题）前文提到的「空 Shape ReLU」案例（`pypto.Tensor([], pypto.DT_FP32)`）说明了什么问题？  
   A. ReLU 算子本身很难实现  
   B. 模型能力不足，无法理解简单的张量声明  
   C. 即使模型能力较强，也可能写出不合规但肉眼难以识别的代码，需要工程机制来拦截  
   D. PyPTO 框架的 API 设计太复杂

**执行以下代码获取答案。**

</div>

In [ ]:
!cat ./answer/05.01_answer.txt

<div align="left">

## 本节小结：一张知识框架图

本节回答一个核心问题：**为什么要给大模型开发算子配备工程体系？** 把知识点汇成下面的框架，方便按图复习。

### 一、问题：仅靠一条 Prompt 开发算子会遇到哪些坑

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">表现</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">为什么隐蔽</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信息过载</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">关键约束（shape、dtype、容差）被遗漏或走样</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">需求复杂时，单段上下文装不下也记不清</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编码与验证不分</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">模型写完代码自己判定「通过」，结论不可采信</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">写和判是同一个主体，天然缺乏客观对照</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">步骤被静默跳过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">规范步骤被悄悄简化、绕过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">没有外部约束，偷懒不会被发现</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态丢失</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">中断后无法恢复，或凭记忆对不上</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">进度与依据没有持久化记录</td></tr></tbody></table>

### 二、答案：五道工程防线

<table style="margin-left:0;border-collapse:collapse"><thead><tr><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">防线</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">解决哪个问题</th><th style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">关键机制</th></tr></thead><tbody><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">智能体分工</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">信息过载</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">每个智能体只加载当前阶段所需的一小部分技能</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">独立验证</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编码与验证不分</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">编写实现者永远不参与结果判定（Verifier 独立裁决）</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">Lint Gate</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">步骤被静默跳过</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">写入关键文件时，自动规则检查强制生效</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态持久化</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">状态丢失</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">双份共享状态文件随时可查、可恢复</td></tr><tr><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">真实 NPU 验收</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">结果不可信</td><td style="border:1px solid #d0d0d0;padding:4px 8px;text-align:left">以真实设备运行结果与实际测试为准，不看总结</td></tr></tbody></table>

### 三、核心思想

可靠性保障从「模型本身」转移到「工程架构」：模型能力决定上限，工程机制兜住下限。模型犯的错误往往不报错、难发现，因此需要制度化的手段来发现缺陷、拦截违规、留存证据——这正是后续几节反复强调「不看总结、只信证据」的根源。

下一节走进 Agent 内部：了解一个总控与八个智能体如何分工协作，以及怎样用 OpenCode 驱动它完成一次真实的算子开发。

</div>